# Clase 3 · 00 · Preflight: el mismo dato, cinco modelos

En las clases 1 y 2 construimos un modelo **relacional** (tablas Silver con claves y joins). En esta práctica tomamos esos mismos clientes, productos y transacciones y los reorganizamos según los modelos NoSQL vistos en teoría:

| Notebook | Modelo | Pregunta de negocio | Motor de referencia |
|---|---|---|---|
| `01_cap_pacelc` | Distribución | ¿Qué pasa con un saldo si se corta la red entre réplicas? | Dynamo, MongoDB |
| `02_clave_valor` | Clave-valor | ¿Cómo sirvo el perfil de un cliente en milisegundos? | Redis, DynamoDB |
| `03_documental` | Documental | ¿Cómo guardo el historial completo de un cliente en un solo objeto? | MongoDB, Elasticsearch |
| `04_grafos` | Grafos | ¿Qué clientes comparten dispositivos con clientes sospechosos? | Neo4j |
| `05_vectorial` | Vectorial | ¿Qué clientes se comportan parecido a un sospechoso? | Pinecone |
| `06_columnar` | Columnar | ¿Por qué Parquet/Delta es rápido para analítica? | Parquet, Delta |

Databricks no es Redis, MongoDB ni Neo4j: **simulamos cada modelo** con Spark, tablas Delta y Python para ver sus ideas centrales sin instalar nada. Los notebooks son **demostrativos**: no hay que modificar código, sólo ejecutarlos en orden, leer y mirar los resultados. Las 20 preguntas de la entrega están en el `README.md` y se responden con lo que muestran.

Este notebook sólo verifica que existan las tablas Silver de la clase 2. No modifica nada.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)

customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
for table in [customers, products, transactions]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá 00_preflight (y la clase 2)."
lab_path = f"{config.volume_path}/nosql_lab"
print(f"Esquema: {config.catalog}.{config.schema} · escala {config.scale}")

In [ ]:
counts = {name: spark.table(table).count() for name, table in
          [("silver_customers", customers), ("silver_products", products), ("silver_transactions", transactions)]}
assert counts["silver_customers"] == config.rows["customers"], "student_id o scale no coinciden con las clases 1 y 2"
assert counts["silver_products"] == config.rows["products"], "student_id o scale no coinciden con las clases 1 y 2"
duplicated = (spark.table(transactions).groupBy("transaction_id").count().where("count > 1").count())
assert duplicated == 0, "silver_transactions tiene transaction_id repetidos: revisá la clase 2"
display(spark.createDataFrame([(k, v) for k, v in counts.items()], ["tabla", "filas"]))

## Punto de partida: el modelo relacional

Mirá las tres tablas. Para responder *"¿qué compró el cliente 42 y desde qué dispositivo?"* hay que unir las tres. Durante la práctica vamos a comparar esta forma de organizar los datos con las alternativas NoSQL.

In [ ]:
display(spark.table(customers).orderBy("customer_id").limit(3))
display(spark.table(products).orderBy("product_id").limit(3))
display(spark.table(transactions).orderBy("transaction_id").limit(3))

In [ ]:
example = spark.sql(f"""
SELECT c.customer_id, c.country, c.segment, t.transaction_id, t.event_ts, t.amount,
       t.payment_channel, t.device_id, p.category
FROM {customers} c
JOIN {transactions} t ON t.customer_id = c.customer_id
JOIN {products} p ON p.product_id = t.product_id
WHERE c.customer_id = 42
ORDER BY t.event_ts DESC
""")
display(example)
print(f"Listo. Los recursos de esta práctica se crean como tablas nosql_* y en {lab_path}/")